In [2]:
import sys
import os
from ftplib import FTP
import tempfile
import gzip
import shutil
from collections import defaultdict
from Bio import SeqIO
import pandas as pd

In [3]:
outdir = "/p/vast1/ibap/chivian1/proj/viral_fams/data/refseq_viral_2025-08-06/genomes"
seen_filepath = "/p/vast1/ibap/chivian1/proj/viral_fams/data/refseq_viral_2025-08-06/seen_assemblies.txt"
combined_outdir = "/p/vast1/ibap/chivian1/proj/viral_fams/data/refseq_viral_2025-08-06/combined"

Download

In [ ]:
def read_seen_species_file(filepath):
    seen = set()
    with open(filepath, "r") as file:
        for line in file:
            seen.add(line.strip())
    return seen


def write_seen_species_file(filepath, seen):
    with open(filepath, "w") as file:
        for assembly in seen:
            file.write(f"{assembly}\n")


if not os.path.exists(outdir):
    os.makedirs(outdir)

# seen = set()
seen = read_seen_species_file(seen_filepath)

print(f"Created directory: {outdir}")

try:
    ftp = FTP('ftp.ncbi.nlm.nih.gov', user="anonymous", passwd="golez1@llnl.gov")
    print("Successfully connected and logged in to FTP server.")
except Exception as e:
    print(f"Error connecting to FTP server: {e}")
    sys.exit(-1)

ftp.cwd('/genomes/refseq/viral')

refseq_species = ftp.nlst()
print(f"{len(refseq_species)} Total")

refseq_species = [assembly for assembly in refseq_species if assembly not in seen]

skipped = list()

for i_species, species in enumerate(refseq_species):
    print(f"{i_species}. {species}")
    
    complete = False
    retries = 0
    while not complete:
        try:
            ftp.cwd(f'/genomes/refseq/viral/{species}')
            if "latest_assembly_versions" in ftp.nlst():
                ftp.cwd(f'/genomes/refseq/viral/{species}/latest_assembly_versions')
                for name, attributes in ftp.mlsd():
                    if attributes.get('type') == 'OS.unix=symlink':
                        genome = name
                        genome_outdir = os.path.join(outdir, genome)
                        os.makedirs(genome_outdir, exist_ok=True)

                        ftp.cwd(f'/genomes/refseq/viral/{species}/latest_assembly_versions/{genome}')
                        for file in ftp.nlst():
                            if any(file.endswith(suffix) for suffix in ["genomic.fna.gz", "genomic.gbff.gz", "genomic.gff.gz", "genomic.gtf.gtf", "protein.faa.gz"]):
                                unzipped_path = os.path.join(genome_outdir, file[:-len(".gz")])

                                if os.path.exists(unzipped_path) and \
                                os.path.isfile(unzipped_path) and \
                                os.path.getsize(unzipped_path) > 0:
                                    continue

                                with tempfile.TemporaryDirectory() as tmpdir:
                                    temp_file = os.path.join(tmpdir, file)
                                    with open(temp_file, 'wb') as f:
                                        # print(f"Downloading {file}...")
                                        ftp.retrbinary(f'RETR {file}', f.write)

                                    unzipped_path = os.path.join(genome_outdir, file[:-len(".gz")])
                                    with gzip.open(temp_file, 'rb') as f_in, open(unzipped_path, 'wb') as f_out:
                                        # print(f"Unzipping {file}...")
                                        shutil.copyfileobj(f_in, f_out)
            
            complete = True

        except Exception as e1:
            retries += 1
            if retries > 10:
                print(f"{e1}\nToo many retries... Skipping.")

                skipped.append(species)
                break
            else:
                print(f"{e1}\nTrying again...")

                try:
                    ftp = FTP('ftp.ncbi.nlm.nih.gov', user="anonymous", passwd="golez1@llnl.gov")
                    print("Successfully connected and logged in to FTP server.")
                except Exception as e2:
                    print(f"Error connecting to FTP server: {e2}")
                    sys.exit(-1)
    
    seen.add(species)


write_seen_species_file(seen_filepath, seen)

In [ ]:
print(skipped)

Create Data Summary File

In [1]:
def read_fasta_to_dict(filepath):
    seq_map = dict()
    with open(filepath, "r") as file:
        lines = file.readlines()
        name = ""
        seq = ""
        for line in lines:
            line = line.strip("*\n")
            if line.startswith(">"):
                if name and seq:
                    seq_map[name] = seq
                name = line[1:]
                seq = ""
            else:
                seq += line
                
        if name and seq:
            seq_map[name] = seq
    
    return seq_map


def write_sequences_to_file(faa_out, faa_map):
    with open(faa_out, "w") as file:
        for name, seq in faa_map.items():
            file.write(f">{name}\n")
            file.write(f"{seq}\n")


def extract_name(attributes_string):
    attributes = attributes_string.split(";")
    for attribute in attributes:
        if attribute.startswith("Name="):
            return attribute.split("=")[1]
    return None
    
        
        

def get_genome_info(assembly_dir):
    faa_map = dict()
    df_info = list()

    with os.scandir(assembly_dir) as assemblies:
        for assembly in assemblies:
            if assembly.is_dir():
                with os.scandir(assembly.path) as files_iterator:
                    info_files = list(files_iterator)
                    if all([any([file.path.endswith(suffix) for file in info_files]) for suffix in ["protein.faa", "genomic.gbff", "genomic.gff"]]):
                        genes = list()
                        genomes = list()
                        starts = list()
                        stops = list()
                        strands = list()
                        taxonomy = None
                        species = None
                        for file in info_files:
                            if file.path.endswith("protein.faa"):
                                fasta_dict = read_fasta_to_dict(file.path)

                                '''
                            elif file.endswith("genomic.fna") and not file.endswith("cds_from_genomic.fna"):
                                fasta_dict = read_fasta_to_dict(filepath)
                                for prot, seq in fasta_dict.items():
                                    nc_seq_map[genome][prot] = seq
                                '''

                            elif file.path.endswith("genomic.gbff"):
                                for record in SeqIO.parse(file.path, "genbank"):
                                    taxonomy = "; ".join(record.annotations["taxonomy"])
                                    species = record.annotations["organism"]

                            elif file.path.endswith("genomic.gff"):
                                with open(file.path) as gff:
                                    lines = gff.readlines()
                                    for line in lines:
                                        line = line.strip()
                                        if not line.startswith("#"):
                                            features = line.split("\t")
                                            if features[2] == "CDS":
                                                genome = features[0]
                                                start = int(features[3]) - 1
                                                stop = int(features[4])
                                                strand = features[6]
                                                gene = extract_name(features[8])    # will not find pseudogenes

                                                genomes.append(genome)
                                                starts.append(start)
                                                stops.append(stop)
                                                strands.append(strand)
                                                genes.append(gene)

                        for i in range(len(genes)):
                            gene = genes[i]
                            genome = genomes[i]
                            start = starts[i]
                            stop = stops[i]
                            strand = strands[i]

                            df_info.append((gene, genome, os.path.basename(assembly.path), start, stop, strand, taxonomy, species))
                        
                        faa_map.update(fasta_dict)

    return pd.DataFrame(df_info, columns=["gene_id", "genome_acc", "assembly", "start", "stop", "strand", "lineage", "species"]), faa_map

if not os.path.exists(combined_outdir):
    os.makedirs(combined_outdir)

df, faa_map = get_genome_info(outdir)
df.to_csv(os.path.join(combined_outdir, "gene_data.tsv"), sep="\t", index=False)
write_sequences_to_file(os.path.join(combined_outdir, "genes.faa"), faa_map)


NameError: name 'os' is not defined

hmmsearch \
-o /p/vast1/ibap/chivian1/proj/viral_fams/data/refseq_viral_2025-08-06/vogdb_vog_231_hmmsearch/vogdb_vog_outfile.txt \
--tblout /p/vast1/ibap/chivian1/proj/viral_fams/data/refseq_viral_2025-08-06/vogdb_vog_231_hmmsearch/vogdb_vog_tbl_out.txt \
--domtblout /p/vast1/ibap/chivian1/proj/viral_fams/data/refseq_viral_2025-08-06/vogdb_vog_231_hmmsearch/vogdb_vog_domtbl_out.txt \
--notextw \
/p/vast1/ibap/chivian1/proj/viral_fams/dbs/VOGDB_vogs_231/vog_combined.hmm \
/p/vast1/ibap/chivian1/proj/viral_fams/data/refseq_viral_2025-08-06/combined/genes.faa